# Q5 — Naive Bayes Classifier (Play Weather Example)

Calculate probability of `Play=Yes` when `Outlook=Overcast` and `Temperature=Mild` using Naive Bayes assumption and also implement a classifier.

In [1]:
import pandas as pd
from collections import defaultdict, Counter
import math

# Play Dataset (same as Q3)
data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
    'Windy': ['False','True','False','False','False','True','True','False','False','False','True','True','False','True'],
    'Play': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
}
df = pd.DataFrame(data)
df.head()

,Outlook,Temperature,Humidity,Windy,Play
0,Sunny,Hot,High,False,No
1,Sunny,Hot,High,True,No
2,Overcast,Hot,High,False,Yes
3,Rain,Mild,High,False,Yes
4,Rain,Cool,Normal,False,Yes


In [5]:
# Compute priors and likelihoods (categorical features) with Laplace smoothing
def train_nb(df, target_col):
    priors = df[target_col].value_counts(normalize=True).to_dict()
    cond_probs = {}
    features = [c for c in df.columns if c != target_col]
    for cls in df[target_col].unique():
        subset = df[df[target_col] == cls]
        cond_probs[cls] = {}
        for feat in features:
            counts = subset[feat].value_counts()
            total = len(subset)
            # Laplace smoothing for categorical features
            values = df[feat].unique()
            prob_dict = {}
            for v in values:
                prob_dict[v] = (counts.get(v,0) + 1) / (total + len(values))
            cond_probs[cls][feat] = prob_dict
    return priors, cond_probs

priors, cond_probs = train_nb(df, 'Play')
print('Priors:', priors)

Priors: {'Yes': 0.6428571428571429, 'No': 0.35714285714285715}


In [3]:
# Compute probability for Outlook=Overcast AND Temperature=Mild
def predict_instance(priors, cond_probs, instance):
    scores = {}
    for cls, prior in priors.items():
        score = math.log(prior)
        for feat, val in instance.items():
            score += math.log(cond_probs[cls][feat].get(val, 1e-9))
        scores[cls] = score
    # return normalized probabilities
    max_log = max(scores.values())
    exp_scores = {cls: math.exp(s - max_log) for cls,s in scores.items()}
    ssum = sum(exp_scores.values())
    probs = {cls: v/ssum for cls,v in exp_scores.items()}
    return probs

instance = {'Outlook':'Overcast', 'Temperature':'Mild'}  # given
probs = predict_instance(priors, cond_probs, instance)
print('Posterior probabilities given instance:', probs)
print('Classified as:', max(probs, key=probs.get))

Posterior probabilities given instance: {'Yes': 0.8695652173913042, 'No': 0.1304347826086957}
Classified as: Yes


This computes P(Play=Yes | Outlook=Overcast, Temperature=Mild) and similarly for No using Naive Bayes with Laplace smoothing.